In [ ]:
import random
from datasets import load_dataset
import os
from typing import Generator, Any

# ======================================================================
# 🌟 데이터셋 소개: Alpaca Military Advisor
# 📝 제목: 군사 장비 어드바이저 데이터셋
# 💡 의미: 영화나 유튜브 비디오의 자막(subtitles)을 기반으로 GPT 모델이
#       '질문(Instruction)'과 '답변(Output)' 쌍을 만들어 학습시킨 데이터셋입니다.
# 🛠️ 사용 목적: 특정 군사 장비(특히 차량, 장비)에 대한 질문과 답변을 통해
#    효율적인 AI Q&A 시스템이나 프롬프트 엔지니어링을 실습하는 데 최적입니다.
# 🧑‍🏫 이번 실습 목표: 이 데이터셋을 탐색하여, 단순한 Q&A 쌍을 실제 LLM이 이해하기 쉬운
#    '프롬프트 템플릿'으로 재구성하는 방법을 배울 거예요!
# ======================================================================

# 상수 설정 (튜터가 편하게 수정할 수 있도록!)
DATASET_NAME = "ilyusha07/alpaca_military_equipment_advisor"
SAMPLE_COUNT = 5  # 😄 재미있게 맛보기로 5개만 살펴볼 거예요!

def load_dataset_with_fallback(dataset_id: str, split: str) -> Any:
    """
    데이터셋 로드 함수: 스트리밍 방식을 우선 시도하고, 실패하면 일반 다운로드 방식으로 전환합니다.
    """
    print("🚀 데이터셋을 로드합니다. (스트리밍 모드 우선 시도)")
    
    try:
        # 1. 스트리밍 모드 시도 (대용량 데이터셋에서 빠르고 메모리 효율적!)
        dataset = load_dataset(dataset_id, split=split, streaming=True)
        print("✅ 스트리밍 모드(streaming=True)로 성공적으로 데이터를 로드했습니다! 👏")
        return dataset
    except Exception as e:
        print(f"🚨 스트리밍 모드 로드 실패 ({type(e).__name__} 오류): {e}")
        print("🔄 일반(다운로드) 모드로 전환하여 로드를 재시도합니다. (작은 샘플만 로드할게요!)")
        try:
            # 2. 일반 다운로드 모드로 전환
            return load_dataset(dataset_id, split=split, streaming=False)
        except Exception as e_fallback:
            print(f"❌ 모든 로드 시도가 실패했습니다. 에러: {e_fallback}")
            raise

def process_and_generate_template(sample: dict) -> str:
    """
    주어진 Q&A 샘플을 LLM에 최적화된 프롬프트 템플릿으로 변환합니다.
    """
    instruction = sample.get("instruction", "")
    output = sample.get("output", "")
    
    # 🧠 AI가 가장 좋아하는 포맷: 역할 정의 + 질문 + 답변 요청!
    template = f"""
[TASK]: 사용자 질문에 대한 전문적인 답변을 하세요. 답변은 주어진 문맥과 데이터 기반이어야 합니다.
[CONTEXT]: (사용할 문맥 정보가 있다면 여기에 넣으세요.)
[QUESTION]: {instruction}
[ANSWER]: {output}
"""
    return template.strip()

# ======================================================================
# ✨ 메인 실습 시작!
# ======================================================================
print("="*80)
print("🤖 안녕하세요! 파이썬 코딩 튜터입니다! 오늘 데이터셋을 분석하며 AI의 비밀을 파헤쳐 봅시다! 🎩")
print("="*80)

# 1. 데이터 로드
try:
    # 'train' 스플릿을 사용하고, 로드 함수를 통해 스트리밍/일반 모드를 처리합니다.
    dataset = load_dataset_with_fallback(DATASET_NAME, split='train')
except Exception:
    print("🛑 데이터셋 로드 실패로 인해 실습을 중단합니다.")
    exit()


# 2. 데이터 샘플링 전략 (Streaming과 Non-streaming을 모두 아우르는 만능 패턴!)
# 💡 튜터의 Tip: 스트리밍 데이터셋(IterableDataset)인지 확인하고, 그에 맞는 방식으로 데이터를 가져와야 해요.
if hasattr(dataset, "take"):
    # case 1: 스트리밍 데이터셋인 경우 (take() 메소드 사용)
    print(f"\n⭐️ 데이터셋은 스트리밍 모드입니다. 상위 {SAMPLE_COUNT}개의 샘플을 순차적으로 가져올게요.")
    # take()를 사용해 Iterator를 생성합니다.
    sample_iterator: Generator[dict, None, None] = dataset.take(SAMPLE_COUNT)
else:
    # case 2: 일반 데이터셋인 경우 (list(dataset.select()) 패턴 사용)
    print(f"\n⭐️ 데이터셋은 일반 모드입니다. 상위 {SAMPLE_COUNT}개의 샘플을 리스트로 가져올게요.")
    # 리스트로 변환하여 샘플을 추출합니다.
    sampled_dataset = dataset.select(range(min(SAMPLE_COUNT, 100)))
    sample_iterator = iter(sampled_dataset)


# 3. 샘플 분석 및 프롬프트 템플릿 생성 실습 (핵심!)
print("\n" + "="*80)
print(f"🔥 실습 목표: {SAMPLE_COUNT}개의 샘플을 분석하여 AI에게 가장 친절한 '프롬프트 템플릿'을 만들어 줍니다.")
print("="*80)

print("\n>>> [샘플 분석 결과] <<<\n")

sample_count = 0
for sample in sample_iterator:
    if sample_count >= SAMPLE_COUNT:
        break
        
    # 데이터 구조 확인 및 출력
    instruction = sample.get("instruction", "[N/A]")
    output = sample.get("output", "[N/A]")

    print(f"====================\n[🔍 {sample_count + 1}번째 샘플 분석]\n")
    print(f"  ➡️ 원본 질문 (Instruction): {instruction[:80].replace('\n', ' ') if len(str(instruction)) > 80 else instruction}")
    print(f"  💡 원본 답변 (Output): {output[:80].replace('\n', ' ') if len(str(output)) > 80 else output}")
    
    # 📝 창의적인 사용 예시: 이를 LLM의 '지시문(Prompt)'으로 재구성
    prompt_template = process_and_generate_template(sample)
    
    print("\n========================================================================================================")
    print("✨ [AI 최적화 프롬프트 템플릿 (출력)] ✨")
    print(prompt_template)
    print("========================================================================================================\n")
    
    sample_count += 1

print("\n🎉 와! 수고하셨어요! 🙌")
print("여기서 우리가 배운 것은, 단순히 데이터('질문', '답변')를 가지고 있는 것이 아니라,")
print("그 데이터를 AI가 가장 잘 이해할 수 있는 '규칙(템플릿)'의 형태로 재가공하는 것이 중요하다는 점이에요!")
print("이것이 바로 데이터 엔지니어링과 프롬프트 엔지니어링의 핵심입니다! 👍")
print("================================================================================")